In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Data/layoffs.csv")

In [3]:
df.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
0,Meta,SF Bay Area,700.0,3/25/2026,0.01,Consumer,https://www.nytimes.com/2026/03/25/technology/...,Post-IPO,26000.0,United States,3/25/2026
1,Epic Games,Raleigh,1000.0,3/24/2026,NaN,Consumer,https://www.epicgames.com/site/en-US/news/toda...,Unknown,6400.0,United States,3/24/2026
2,OpenText,"Waterloo, Non-U.S.",NaN,3/24/2026,NaN,Data,https://betakit.com/opentext-employees-report-...,Post-IPO,1100.0,Canada,3/25/2026
3,Spotify,"Stockholm, Non-U.S.",15.0,3/23/2026,NaN,Media,https://variety.com/2026/digital/news/spotify-...,Post-IPO,2100.0,Sweden,3/25/2026
4,Gemini,New York City,NaN,3/20/2026,0.30,Crypto,https://www.bloomberg.com/news/articles/2026-0...,Unknown,423.0,United States,3/22/2026


In [6]:
df.shape

(4333, 11)

In [7]:
df.dtypes

company                 object
location                object
total_laid_off         float64
date                    object
percentage_laid_off    float64
industry                object
source                  object
stage                   object
funds_raised           float64
country                 object
date_added              object
dtype: object

In [4]:
df.columns

Index(['company', 'location', 'total_laid_off', 'date', 'percentage_laid_off',
       'industry', 'source', 'stage', 'funds_raised', 'country', 'date_added'],
      dtype='object')

In [5]:
df.isnull().sum()

company                   0
location                  1
total_laid_off         1495
date                      0
percentage_laid_off    1606
industry                  2
source                    3
stage                     5
funds_raised            493
country                   2
date_added                0
dtype: int64

In [6]:
df = df.drop(columns=['location', 'source', 'date_added'])


In [7]:
df.columns

Index(['company', 'total_laid_off', 'date', 'percentage_laid_off', 'industry',
       'stage', 'funds_raised', 'country'],
      dtype='object')

In [8]:
df.describe()

,total_laid_off,percentage_laid_off,funds_raised
count,2838.000000,2727.000000,3840.000000
mean,297.649753,0.293552,850.863202
std,1040.029024,0.303430,4570.174088
min,3.000000,0.000000,0.700000
25%,40.000000,0.100000,54.000000
50%,90.000000,0.170000,174.000000
75%,200.000000,0.330000,483.250000
max,22000.000000,1.000000,121900.000000


#### **Finding Missing Values**

In [9]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': round(df.isnull().sum() / len(df) * 100, 2)
})
print(missing[missing['Missing Count'] > 0])

                     Missing Count  Missing %
total_laid_off                1495      34.50
percentage_laid_off           1606      37.06
industry                         2       0.05
stage                            5       0.12
funds_raised                   493      11.38
country                          2       0.05


#### **Remove Duplicates**

In [10]:
print("Before duplicates removal:", df.shape)

Before duplicates removal: (4333, 8)


In [11]:
duplicates = df[df.duplicated()]
print(f"\nDuplicate rows found: {len(duplicates)}")
print(duplicates.head())


Duplicate rows found: 3
          company  total_laid_off        date  percentage_laid_off  \
280        Cars24           200.0   4/26/2025                  NaN   
2855  Beyond Meat           200.0  10/14/2022                 0.19   
3466        Cazoo           750.0    6/7/2022                 0.15   

            industry     stage  funds_raised         country  
280   Transportation  Series G        1300.0           India  
2855            Food  Post-IPO         122.0   United States  
3466  Transportation  Post-IPO        2000.0  United Kingdom  


In [19]:
df = df.drop_duplicates()
print("\nAfter removal:", df.shape)


After removal: (4330, 8)


#### **Fix the Date Column**

In [12]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')   
df['year_month'] = df['date'].dt.to_period('M')   

print("Date range of dataset:")
print("From:", df['date'].min(), "→ To:", df['date'].max())
print("\nLayoffs recorded per year:\n", df['year'].value_counts().sort_index())


Date range of dataset:
From: 2020-03-11 00:00:00 → To: 2026-03-25 00:00:00

Layoffs recorded per year:
 year
2020     635
2021      44
2022    1230
2023    1387
2024     633
2025     327
2026      77
Name: count, dtype: int64


#### **Clean Messy Text**

In [13]:
# Strip spaces, fix casing
df['company'] = df['company'].str.strip()
df['industry'] = df['industry'].str.strip().str.title()
df['country'] = df['country'].str.strip()

# Fix known dirty value in this dataset
df['country'] = df['country'].str.replace('United States.', 'United States', regex=False)



In [14]:
# See unique industries after cleaning
print("Unique Industries:\n", sorted(df['industry'].dropna().unique()))

Unique Industries:
 ['Aerospace', 'Ai', 'Construction', 'Consumer', 'Crypto', 'Data', 'Education', 'Energy', 'Finance', 'Fitness', 'Food', 'Hardware', 'Healthcare', 'Hr', 'Infrastructure', 'Legal', 'Logistics', 'Manufacturing', 'Marketing', 'Media', 'Other', 'Product', 'Real Estate', 'Recruiting', 'Retail', 'Sales', 'Security', 'Support', 'Transportation', 'Travel']


#### **Drop Rows That Are Truly Useless**

In [15]:
print("Before dropping empty layoff rows:", df.shape)

# If BOTH total_laid_off AND percentage_laid_off are null → useless row
df = df[~(df['total_laid_off'].isnull() & df['percentage_laid_off'].isnull())]

print("After dropping:", df.shape)
print(f"\nRows removed: {len(df) - df.shape[0]}")

Before dropping empty layoff rows: (4333, 12)
After dropping: (3623, 12)

Rows removed: 0


####  **Final Data Check Before Saving**

In [16]:
print("=== FINAL DATA CHECK ===")
print(f"Total Rows     : {df.shape[0]}")
print(f"Total Columns  : {df.shape[1]}")
print(f"Date Range     : {df['date'].min()} → {df['date'].max()}")
print(f"Countries      : {df['country'].nunique()}")
print(f"Companies      : {df['company'].nunique()}")
print(f"Industries     : {df['industry'].nunique()}")
print(f"Missing Values :\n{df.isnull().sum()[df.isnull().sum() > 0]}")

=== FINAL DATA CHECK ===
Total Rows     : 3623
Total Columns  : 12
Date Range     : 2020-03-11 00:00:00 → 2026-03-25 00:00:00
Countries      : 61
Companies      : 2505
Industries     : 30
Missing Values :
total_laid_off         785
percentage_laid_off    896
industry                 2
stage                    4
funds_raised           395
country                  2
dtype: int64


#### **Save Cleaned Data**

In [17]:
df.columns

Index(['company', 'total_laid_off', 'date', 'percentage_laid_off', 'industry',
       'stage', 'funds_raised', 'country', 'year', 'month', 'month_name',
       'year_month'],
      dtype='object')

In [21]:
# Check nulls before
print("Nulls before:")
print(df[['total_laid_off', 'funds_raised']].isnull().sum())

# Fix only these 2 columns
df['total_laid_off'] = df['total_laid_off'].fillna(0).astype(int)
df['funds_raised'] = df['funds_raised'].fillna(0)

# Verify
print("\nNulls after:")
print(df[['total_laid_off', 'funds_raised']].isnull().sum())

Nulls before:
total_laid_off    785
funds_raised      395
dtype: int64

Nulls after:
total_laid_off    0
funds_raised      0
dtype: int64


In [22]:
df.to_csv("data/layoffs_cleaned.csv", index=False)
print("✅ Fixed and saved!")

✅ Fixed and saved!
